# Milestone 4 lab, notebook 3 of 7: Clean, join, enrich

## Where we are

| Notebook | What you do |
|---|---|
| 1. Load first, transform later | Python stops transforming. Compare ELT with milestone 1's ETL |
| 2. Your first model | Write one SELECT in a file, and let dbt build it |
| **3. Clean, join, enrich** (you are here) | One row per order again, then two joins and their traps |
| 4. A loop or a set | Answer one question row by row, then with one statement |
| 5. Layers and the incremental model | Three layers, and a report that rebuilds only its newest days |
| 6. Tests on the tables | Break the data on purpose, and watch the pipeline refuse it |
| 7. The production version | One morning as ELT: Python loads, dbt transforms and tests |

A clean column is not yet a clean table. The raw table still holds every version of every order. This notebook makes one row per order in SQL, then joins tables, and meets two ways a join can quietly ruin a number.

In [1]:
from lab_helpers import dbt, lab_warehouse, load_only, money, show_doc, table

con = lab_warehouse(fresh=True)
load_only(con, "2026-06-08", "2026-06-09")
dbt("build", show=False)
print("Raw rows are landed, and the models are built.")

Landed 226 raw rows for the morning of 2026-06-08.
Landed 102 raw rows for the morning of 2026-06-09.


Raw rows are landed, and the models are built.


## Step 1: Deduplication

In milestone 3 a `MERGE` kept one row per order. Here the same result comes from one `SELECT`. Number the versions of each order, newest first.

In [2]:
table(con, """
    SELECT (payload->>'order_id')::bigint AS order_id,
           payload->>'status' AS status,
           (payload->>'refunded_cents')::int AS refunded_cents,
           payload->>'updated_at' AS updated_at,
           row_number() OVER (
               PARTITION BY (payload->>'order_id')::bigint
               ORDER BY (payload->>'updated_at')::timestamptz DESC, landing_id DESC
           ) AS version_rank
    FROM raw.orders_landing
    WHERE (payload->>'order_id')::bigint IN (1130, 1224)
    ORDER BY 1, version_rank
""")

,order_id,status,refunded_cents,updated_at,version_rank
0,1130,delivered,578,2026-06-08T16:00:00Z,1
1,1130,delivered,0,2026-06-05T20:45:00Z,2
2,1224,delivered,0,2026-06-08T15:25:00Z,1
3,1224,placed,0,2026-06-08T03:34:57Z,2


`PARTITION BY order_id` starts the numbering again for each order. `ORDER BY updated_at DESC` puts the newest version first. So the newest version of every order has `version_rank = 1`.

Keep only those rows. That is the last step of `stg_orders`: `where version_rank = 1`.

In [3]:
table(con, """
    SELECT (SELECT count(*) FROM raw.orders_landing)    AS raw_rows,
           (SELECT count(*) FROM staging.stg_orders)    AS rows_after_dedupe,
           (SELECT count(DISTINCT order_id) FROM staging.stg_orders) AS different_orders
""")

,raw_rows,rows_after_dedupe,different_orders
0,328,251,251


328 rows went in. 251 came out, one per order. Removing repeats like this is called **deduplication**.

Notice what is missing: there is no `MERGE`, no primary key and no "is this newer?" check. The model reads **all** the raw rows each time and keeps the newest. Run it once or ten times, and it gives the same table. It is idempotent because it is a rebuild, not an update.

## Step 2: A join

The orders table has a city **code**: `NGT`. The manager wants the city's **name**. The names are in another small table.

In [4]:
table(con, "SELECT * FROM core.cities")

,city,city_name
0,NGT,Northgate
1,RVB,Riverbend
2,LKS,Lakeshore


That table is a **seed**: a CSV file kept inside the dbt project, for small lists that people maintain by hand.

A **join** puts two tables side by side where a key matches.

In [5]:
table(con, """
    SELECT o.order_id, o.city, c.city_name, o.status, o.total_cents
    FROM staging.stg_orders AS o
    LEFT JOIN core.cities AS c ON c.city = o.city
    WHERE o.order_id IN (1130, 1196, 1197)
""")

,order_id,city,city_name,status,total_cents
0,1130,NGT,Northgate,delivered,1475
1,1196,RVB,Riverbend,delivered,1796
2,1197,LKS,Lakeshore,delivered,1904


Each order gained a column from another table. Adding columns that way is called **enrichment**.

## Step 3: Trap one, the join that loses rows

Why `LEFT JOIN`, and not a plain `JOIN`? Imagine Arden opens a fourth city, and nobody adds it to the list. Take one city out of the list to see what each join does.

In [6]:
con.execute("CREATE TEMP TABLE cities_short AS SELECT * FROM core.cities WHERE city <> 'LKS'")

table(con, """
    SELECT 'JOIN (inner)' AS how, count(*) AS orders, sum(total_cents) AS total_cents
    FROM staging.stg_orders AS o JOIN cities_short AS c ON c.city = o.city
    UNION ALL
    SELECT 'LEFT JOIN', count(*), sum(total_cents)
    FROM staging.stg_orders AS o LEFT JOIN cities_short AS c ON c.city = o.city
""")

,how,orders,total_cents
0,JOIN (inner),184,450461
1,LEFT JOIN,251,609005


The inner join **dropped every Lakeshore order**, and showed no error. The left join kept them, with an empty city name.

An empty name is a problem you can find. A missing order is not. So: **count the rows before and after every join.** In notebook 6 a test will look for that empty name.

## Step 4: Trap two, the join that multiplies rows

Now join orders to their items, and add up the order totals.

In [7]:
table(con, """
    SELECT 'orders alone' AS what, count(*) AS rows, sum(total_cents) AS sum_of_total_cents
    FROM core.orders WHERE order_date = '2026-06-07'
    UNION ALL
    SELECT 'orders joined to items', count(*), sum(o.total_cents)
    FROM core.orders AS o JOIN core.order_items AS i USING (order_id)
    WHERE o.order_date = '2026-06-07'
""")

,what,rows,sum_of_total_cents
0,orders alone,30,80711
1,orders joined to items,110,319877


30 orders became 110 rows, and Sunday's total is almost four times too big. Each order now appears once **per item**, and its total was added once per item.

You met this number in milestone 1, notebook 3. It is the grain mistake. A join can change the grain of a result without telling you. This one is called a **fan-out**.

The cure is the same as then: say what one row means. If you want money per order, add up at order grain. If you want units per product, add up the item's own columns:

In [8]:
table(con, """
    SELECT sum(i.line_total_cents) AS sum_of_item_lines
    FROM core.orders AS o JOIN core.order_items AS i USING (order_id)
    WHERE o.order_date = '2026-06-07' AND o.status <> 'cancelled'
""")

,sum_of_item_lines
0,80711


The item lines themselves add up to the right $807.11.

## Step 5: The model that does the joins

In [9]:
from lab_helpers import MILESTONE_DIR
print((MILESTONE_DIR / "dbt_project" / "models" / "core" / "orders.sql").read_text())

-- Core: the clean orders table. One row per order, with its latest state.
-- It is rebuilt from the raw data on every run. That is possible because raw keeps everything.

select
    o.order_id,
    o.customer_id,
    o.city,
    c.city_name,                                  -- enrichment: a column added from another table
    o.status,
    o.placed_at,
    o.order_date,
    o.updated_at,
    o.delivered_at,
    o.cancelled_at,
    o.total_cents,
    o.refunded_cents,
    o.total_cents - o.refunded_cents              as net_cents,
    (d.order_id is not null)                      as is_deleted,     -- soft delete: the row stays, marked
    d.deleted_at,
    o.landing_id,
    o.run_id,
    o.loaded_at
from {{ ref('stg_orders') }} as o
left join {{ ref('cities') }} as c                -- left join: an unknown city must not lose the order
    on c.city = o.city
left join {{ ref('stg_order_deletes') }} as d
    on d.order_id = o.order_id



`{{ ref('stg_orders') }}` means "the model called `stg_orders`". It does two jobs. It writes the real table name. And it tells dbt that `orders` must be built **after** `stg_orders`. You never write the order of the steps yourself.

In [10]:
con.close()
show_doc("LLD.md", "4. Core: join and enrich (`orders.sql`)")

**From `docs/LLD.md`**

> ## 4. Core: join and enrich (`orders.sql`)
> 
> ```sql
> from {{ ref('stg_orders') }} as o
> left join {{ ref('cities') }} as c on c.city = o.city                      -- enrichment
> left join {{ ref('stg_order_deletes') }} as d on d.order_id = o.order_id   -- soft delete
> ```
> 
> | Column added | From | The word |
> |---|---|---|
> | `city_name` | The `cities` seed | **Enrichment**: adding columns from another table |
> | `is_deleted`, `deleted_at` | `stg_order_deletes` | The soft delete of milestone 3, now worked out in SQL |
> | `net_cents` | `total_cents - refunded_cents` | A value defined once, used by every model after it |
> 
> Both joins are **left joins**. An order in a city the seed does not know must stay in the table. Then a test can find it. An inner join would drop it with no error.

## New words

| Word | Meaning | Here |
|---|---|---|
| **Deduplication** | Keeping one row per key. "For each key, keep the newest row." | `row_number() ... where version_rank = 1`: 328 rows become 251 |
| **Join** | "A join puts two tables side by side where a key matches." | Orders with cities, orders with items |
| **Enrichment** | "To enrich is to add columns from another table." | `city_name`, added from the `cities` seed |

The definitions are from the D8Loop chapter [Transformation Fundamentals](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/transformation-fundamentals).

## What you now know

- `row_number()` over each order, newest first, then keep number 1: one row per order.
- A left join keeps a row with no match. An inner join drops it, with no error.
- Joining to a table with more rows per key multiplies your rows. Check the grain before you add up.
- `ref()` names another model, and gives dbt the order to build in.

**Say it in an interview:** "I count rows before and after every join. An inner join to a reference table once would have dropped a whole city's orders silently, so I use a left join and a test for the missing value. And I never sum an order-level amount after joining to items: that fan-out made one day's revenue four times too large."

Next: [notebook 4](04_a_loop_or_a_set.ipynb) asks how the work should be done: one row at a time, or all at once.